In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -U math-verify

In [ ]:
import subprocess
import sys

def install_if_missing(package, import_name=None):
    """Install package if not already available."""
    try:
        __import__(import_name or package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

# Core dependencies
install_if_missing("transformers")
install_if_missing("peft")
install_if_missing("accelerate")
install_if_missing("datasets")
install_if_missing("trl")
install_if_missing("tqdm")
install_if_missing("wandb")

print("\n" + "=" * 70)
print("✅ All packages ready!")
print("=" * 70)

# Verify GPU
import torch
print(f"\n GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from huggingface_hub import list_repo_files, hf_hub_download
from datetime import datetime
from peft import PeftModel, get_peft_model, prepare_model_for_kbit_training, LoraConfig, TaskType
from huggingface_hub import snapshot_download
import wandb
import os, re, json, shutil
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login as hf_login
from bitsandbytes.optim import AdamW8bit
import random
from math_verify import LatexExtractionConfig, parse, verify
from math_verify.parser import LatexExtractionConfig, ExprExtractionConfig
import signal
from typing import Optional

In [ ]:
# ========================= CONFIG =========================
STUDENT_ID = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

SEED = 42
GROUP_SIZE = 10
PROMPTS_PER_STEP = 2
MAX_NEW_TOKENS = 512
MAX_STEPS = 100
TEMPERATURE = 1.2
TOP_P = 0.99
LR = 1e-4
CLIP_EPS = 0.2
CHUNK_SIZE = 1 
GRAD_ACCUM_STEPS= 4
GRADE_TIMEOUT_SECONDS = 5
TRAIN_MICROBATCH_ROWS = 2
#MAX_TOTAL_LEN = 512

In [ ]:
# ---- LoRA ----
LORA_R = 64
LORA_ALPHA = 128
LORA_TARGET = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.0

In [ ]:
# ---- Checkpointing ----
CKPT_DIR = "./gating_distilled_rl_checkpoint"
BEST_DIR = os.path.join(CKPT_DIR, "best")
LOG_FILE = "./gating_distilled_rl_training_log.jsonl"
CKPT_EVERY = 20
VAL_EVERY = 20
VAL_BATCH_SIZE = 40
MAX_VAL_SAMPLES = 200

USE_WANDB = True
PUSH_TO_HUB = True
HF_HUB_ID = None

os.makedirs(CKPT_DIR, exist_ok=True)
WANDB_PROJECT = "llama-distilled-rl-V12-gated"

In [ ]:
# ========================= AUTH =========================
HF_TOKEN = None
WANDB_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try: HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except: pass
    try: WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")
    except: pass
except:
    pass

if not HF_TOKEN: HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN: raise RuntimeError("HF_TOKEN missing.")
if not WANDB_API_KEY: WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
if USE_WANDB and WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    USE_WANDB = False
    print("W&B disabled.")

hf_login(token=HF_TOKEN)
torch.manual_seed(SEED)

In [ ]:
ENTROPY_THRESHOLD = 1.57
GATE_TEMP         = 0.35

print("=" * 70)
print("V12-gated: confidence gate constants")
print(f"  ENTROPY_THRESHOLD = {ENTROPY_THRESHOLD}")
print(f"  GATE_TEMP         = {GATE_TEMP}")
print(f"  Source: T=1.2 calibration, p75(H) and std(H)/4")
print("=" * 70)

In [ ]:
# =============================================================================
# DistilledRL V12-gated
# -----------------------------------------------------------------------------
# Identical to V12 (DRL) in every way except the teacher-weight formula:
#
#   V12:    w = rho_tilde                for A > 0
#   V12-g:  w = 1 + c * (rho_tilde - 1)  for A > 0
#                c = sigmoid((H_star - H_teacher) / tau)
#
# H_teacher is the teacher's per-token distribution entropy (nats).
# H_star = 1.57, tau = 0.35 (T=1.2 calibration: p75(H) and std(H)/4).
#
# Everything else - data pool, model loading, MAX_TOTAL_LEN measurement,
# dynamic sampling, true gradient accumulation across GRAD_ACCUM_STEPS
# fresh-rollout batches, TRAIN_MICROBATCH_ROWS chunking, checkpointing,
# validation - matches V12 exactly.
# =============================================================================
# ========================= HELPERS =========================
def find_latest_checkpoint():
    if os.path.exists(CKPT_DIR):
        ckpts = []
        for d in os.listdir(CKPT_DIR):
            if d.startswith("step"):
                try:
                    num = int(d.replace("step", ""))
                    if os.path.exists(os.path.join(CKPT_DIR, d, "training_state.json")):
                        ckpts.append((num, d))
                except Exception:
                    continue
        if ckpts:
            latest_num, latest_name = sorted(ckpts)[-1]
            return os.path.join(CKPT_DIR, latest_name), latest_num

    print(f"  No local checkpoint. Scanning HF: {HF_HUB_ID}")
    try:
        files = list_repo_files(HF_HUB_ID, token=HF_TOKEN)
        steps = []
        for f in files:
            m = re.match(r"step(\d+)/", f)
            if m:
                steps.append(int(m.group(1)))
        if not steps:
            print("  No checkpoints on HF.")
            return None, 0
        latest_step = max(steps)
        print(f"  Found HF step{latest_step}, downloading...")
        local_path = os.path.join(CKPT_DIR, f"step{latest_step}")
        os.makedirs(local_path, exist_ok=True)
        for fname in ["adapter_config.json", "adapter_model.safetensors",
                      "optimizer.pt", "training_state.json", "rng.pt", "training_log.jsonl"]:
            try:
                hf_hub_download(repo_id=HF_HUB_ID,
                                filename=f"step{latest_step}/{fname}",
                                local_dir=CKPT_DIR, token=HF_TOKEN)
            except Exception:
                if fname == "adapter_model.safetensors":
                    try:
                        hf_hub_download(repo_id=HF_HUB_ID,
                                        filename=f"step{latest_step}/adapter_model.bin",
                                        local_dir=CKPT_DIR, token=HF_TOKEN)
                    except Exception:
                        print("  Warning: no adapter_model")
                elif fname != "training_log.jsonl":
                    print(f"  Warning: could not download {fname}")
        return local_path, latest_step
    except Exception as e:
        print(f"  Error downloading from HF: {e}")
        return None, 0

def split_reasoning(text):
    if "<think" in text and "</think" in text:
        open_tag = re.search(r"<think[^>]*>", text)
        close_tag = re.search(r"</think\s*>", text)
        if open_tag and close_tag and close_tag.start() > open_tag.end():
            return (text[open_tag.end():close_tag.start()].strip(),
                    text[close_tag.end():].strip())
    return None, text

def extract_boxed_content(text):
    idx = text.rfind("\\boxed")
    if idx == -1:
        return None
    i = text.find("{", idx)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i + 1:j]
    return None

def parse_number(s):
    if s is None:
        return None
    s = s.replace(",", "").replace("$", "").strip()
    m = re.findall(r"-?\d+\.?\d*/\d+\.?\d*|-?\d+\.?\d*", s)
    if not m:
        return None
    tok = m[-1]
    try:
        if "/" in tok:
            num, den = tok.split("/")
            return float(num) / float(den)
        return float(tok)
    except Exception:
        return None

def extract_answer(text):
    reasoning, answer_region = split_reasoning(text)
    if reasoning is None:
        answer_region = text
    boxed = extract_boxed_content(answer_region)
    if boxed is not None:
        return parse_number(boxed)
    return None

class _GradeTimeout(Exception):
    pass

def _grade_timeout_handler(signum, frame):
    raise _GradeTimeout()

def extract_answer_regex(text):
    if not text:
        return None
    pats = [
        r"(?:final\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:the\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:therefore|thus|so)\s*,?\s*(?:the\s+)?answer\s*(?:is|=|:)\s*([^\n.]+)",
    ]
    for pat in pats:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            v = parse_number(matches[-1])
            if v is not None:
                return v
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if sentences:
        v = parse_number(sentences[-1])
        if v is not None:
            return v
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if nums:
        v = parse_number(nums[-1])
        if v is not None:
            return v
    return None

def compute_reward(completion, gt_answer):
    signal.signal(signal.SIGALRM, _grade_timeout_handler)
    signal.alarm(GRADE_TIMEOUT_SECONDS)
    try:
        pred_expr = parse(completion,
                          extraction_config=[ExprExtractionConfig(), LatexExtractionConfig()])
        gt_wrapped = (gt_answer if ("\\boxed" in gt_answer or "$" in gt_answer)
                      else f"\\boxed{{{gt_answer}}}")
        gt_expr = parse(gt_wrapped,
                        extraction_config=[ExprExtractionConfig(), LatexExtractionConfig()])
        if pred_expr and gt_expr:
            return 1.0 if verify(gt_expr, pred_expr) else 0.0
    except _GradeTimeout:
        print(f"  [compute_reward] TIMEOUT")
        return 0.0
    except Exception:
        pass
    finally:
        signal.alarm(0)

    pred_num = extract_answer_regex(completion)
    gt_num = parse_number(gt_answer)
    if pred_num is not None and gt_num is not None:
        return 1.0 if abs(pred_num - gt_num) < 1e-4 else 0.0
    return 0.0

def _math_verify_parseable(s):
    try:
        wrapped = s if ("\\boxed" in s or "$" in s) else f"\\boxed{{{s}}}"
        return len(parse(wrapped,
                         extraction_config=[LatexExtractionConfig(boxed_match_priority=0)])) > 0
    except Exception:
        return False

# ========================= TOKENIZER =========================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ========================= DATA POOL =========================
print("Loading data pool...")

def load_math_v8(limit=5000):
    print("  Loading MATH (Levels 3-5)...")
    try:
        ds = load_dataset("qwedsacf/competition_math", split="train", token=HF_TOKEN)
    except Exception:
        ds = load_dataset("hendrycks/competition_math", split="train", token=HF_TOKEN)

    def level_filter(ex):
        lv = ex['level']
        if isinstance(lv, str):
            m = re.search(r'(\d+)', lv)
            return int(m.group(1)) >= 3 if m else False
        return int(lv) >= 3

    ds = ds.filter(level_filter)
    ds = ds.filter(lambda ex: len(ex["solution"]) < 800)
    ds = ds.filter(lambda ex: len(ex["problem"]) < 800)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    data = []
    for ex in ds:
        gt = extract_boxed_content(ex["solution"])
        if gt is None: continue
        if not _math_verify_parseable(gt): continue
        data.append({"question": ex["problem"], "answer": gt, "source": "MATH"})
    print(f"    Loaded {len(data)} MATH.")
    return data

def load_gsm8k_v8(limit=6000):
    print("  Loading GSM8K...")
    ds = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    data = []
    for ex in ds:
        gt = ex["answer"].split("####")[-1].replace(",", "").strip()
        if parse_number(gt) is None: continue
        data.append({"question": ex["question"], "answer": gt, "source": "GSM8K"})
    print(f"    Loaded {len(data)} GSM8K.")
    return data

v8_pool = []
# v8_pool.extend(load_math_v8(5000))    # match V12: GSM8K only
v8_pool.extend(load_gsm8k_v8(6000))
print(f"Total V8 pool size: {len(v8_pool)} problems")

full_train = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
split = full_train.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split["test"]

# ========================= MODELS =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

print("Loading student on cuda:0...")
student_base = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_config, device_map={"": "cuda:0"},
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
student_base = prepare_model_for_kbit_training(student_base)

latest_ckpt, start_step = find_latest_checkpoint()
best_val_score = 0.0
best_step = 0
wandb_run_id = None
base_val_score = 0.4850

if latest_ckpt:
    print(f"Resuming from: {latest_ckpt}")
    student = PeftModel.from_pretrained(student_base, latest_ckpt, is_trainable=True)
    state_path = os.path.join(latest_ckpt, "training_state.json")
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
        start_step     = state.get("step", 0)
        best_val_score = state.get("best_val_score", 0.0)
        best_step      = state.get("best_step", 0)
        wandb_run_id   = state.get("wandb_run_id", None)
        base_val_score = state.get("base_val_score", 0.4850)
        print(f"  Resumed step={start_step}, best_val={best_val_score:.4f}")
else:
    print("Starting fresh LoRA...")
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET,
        lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    student = get_peft_model(student_base, lora_config)

student.print_trainable_parameters()
student.enable_input_require_grads()
student.gradient_checkpointing_enable()
student.config.use_cache = False

optimizer = AdamW8bit(filter(lambda p: p.requires_grad, student.parameters()), lr=LR)
if latest_ckpt:
    opt_path = os.path.join(latest_ckpt, "optimizer.pt")
    if os.path.exists(opt_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location="cuda:0"))

print("Loading teacher on cuda:1...")
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, quantization_config=bnb_config, device_map={"": "cuda:1"},
    torch_dtype=torch.float16, trust_remote_code=True,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f"GPU0: {torch.cuda.memory_allocated('cuda:0')/1e9:.2f}GB, "
      f"GPU1: {torch.cuda.memory_allocated('cuda:1')/1e9:.2f}GB")

# ========================= CORE FUNCTIONS =========================
def get_token_logps(model, input_ids, attention_mask, prompt_lens,
                    return_entropy=False, return_token_entropy=False, device=None):
    """
    Backward-compatible with V12's signature.
      return_entropy=True         -> scalar average (unchanged)
      return_token_entropy=True   -> per-token tensor (used by the gate)

    Returns tuple in order: [logps, comp_mask, (avg_entropy?), (tok_entropy?)]
    """
    if device is not None:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)
    else:
        device = next(model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)

    all_logps, all_masks = [], []
    entropy_sum, entropy_count = 0.0, 0
    all_tok_ent = [] if return_token_entropy else None

    for i in range(0, input_ids.shape[0], CHUNK_SIZE):
        cids  = input_ids[i:i+CHUNK_SIZE]
        cmask = attention_mask[i:i+CHUNK_SIZE]
        cplen = prompt_lens[i:i+CHUNK_SIZE]

        out = model(input_ids=cids, attention_mask=cmask)
        logits  = out.logits[:, :-1, :]
        targets = cids[:, 1:]

        tgt = torch.gather(logits, 2, targets.unsqueeze(-1)).squeeze(-1)
        lse = torch.logsumexp(logits, dim=-1)
        token_logps = tgt - lse

        seq_len = cids.shape[1] - 1
        pos = torch.arange(seq_len, device=device).unsqueeze(0)
        comp_mask = (pos >= (cplen.unsqueeze(1) - 1)).long() * cmask[:, 1:]

        if return_entropy or return_token_entropy:
            probs = torch.softmax(logits.float(), dim=-1)
            tok_ent = -(probs * torch.log(probs.clamp_min(1e-10))).sum(dim=-1)
            if return_entropy:
                entropy_sum   += (tok_ent * comp_mask).sum().item()
                entropy_count += comp_mask.sum().item()
            if return_token_entropy:
                all_tok_ent.append(tok_ent)

        all_logps.append(token_logps)
        all_masks.append(comp_mask)
        del out, logits

    token_logps = torch.cat(all_logps, dim=0).float()
    comp_mask   = torch.cat(all_masks, dim=0)

    result = [token_logps, comp_mask]
    if return_entropy:
        result.append(entropy_sum / max(entropy_count, 1))
    if return_token_entropy:
        result.append(torch.cat(all_tok_ent, dim=0))
    return tuple(result)

def build_prompt(question):
    user_content = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}"
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        add_generation_prompt=True, tokenize=False,
    )

# ========================= DYNAMIC MAX_TOTAL_LEN =========================
print("Measuring prompt length distribution...")
_prompt_token_lens = [
    len(tokenizer(build_prompt(ex["question"]), add_special_tokens=False).input_ids)
    for ex in v8_pool
]
_prompt_token_lens.sort()
_p99_prompt_len = _prompt_token_lens[int(0.99 * (len(_prompt_token_lens) - 1))]
MAX_TOTAL_LEN = _p99_prompt_len + MAX_NEW_TOKENS
print(f"  min={_prompt_token_lens[0]}, median={_prompt_token_lens[len(_prompt_token_lens)//2]}, "
      f"p99={_p99_prompt_len}, max={_prompt_token_lens[-1]}")
print(f"  MAX_TOTAL_LEN = {MAX_TOTAL_LEN}")

def collect_rollouts(examples):
    rows, prompt_lens_list, rewards, group_ids, sources = [], [], [], [], []
    box_hits = 0
    student.eval()

    for gi, ex in enumerate(examples):
        prompt_text = build_prompt(ex["question"])
        prompt_ids = tokenizer(prompt_text, return_tensors="pt",
                               add_special_tokens=False).input_ids.to("cuda:0")
        plen = prompt_ids.shape[1]

        with torch.no_grad():
            gen_inputs = prompt_ids.repeat(GROUP_SIZE, 1)
            gen_attn = torch.ones_like(gen_inputs)
            generated = student.generate(
                gen_inputs, attention_mask=gen_attn,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                pad_token_id=tokenizer.pad_token_id,
            )

        raw_answer = ex["answer"]
        if ex.get("source") == "GSM8K":
            gt_answer = raw_answer.split("####")[-1].replace(",", "").strip()
        else:
            gt_answer = raw_answer.strip()

        for row in generated:
            truncated_row = row[:MAX_TOTAL_LEN]
            rows.append(truncated_row.clone())
            prompt_lens_list.append(plen)
            text = tokenizer.decode(truncated_row[plen:], skip_special_tokens=True)
            rewards.append(compute_reward(text, gt_answer))
            group_ids.append(gi)
            sources.append(ex.get("source", "unknown"))
            if extract_boxed_content(text) is not None:
                box_hits += 1

        del generated, gen_inputs
        torch.cuda.empty_cache()

    box_rate = box_hits / max(len(rows), 1)
    return rows, prompt_lens_list, rewards, group_ids, sources, box_rate

def group_normalize(rewards, group_ids):
    advantages = torch.zeros_like(rewards)
    for g in group_ids.unique():
        mask = group_ids == g
        r = rewards[mask]
        std = r.std()
        if std > 0:
            advantages[mask] = (r - r.mean()) / (std + 1e-4)
    return advantages

def is_informative_group(rewards, group_ids):
    for g in group_ids.unique():
        mask = group_ids == g
        if rewards[mask].std() > 0:
            return True
    return False

def get_informative_group(max_attempts=10):
    for _ in range(max_attempts):
        indices = torch.randint(0, len(v8_pool), (PROMPTS_PER_STEP,))
        examples = [v8_pool[int(i)] for i in indices]

        student.eval()
        rows, plens, rewards, gids, sources, box_rate = collect_rollouts(examples)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device="cuda:0")
        gids_t    = torch.tensor(gids,    device="cuda:0")

        if is_informative_group(rewards_t, gids_t):
            return rows, plens, rewards_t, gids_t, sources, box_rate

        del rows, plens, rewards, gids
        torch.cuda.empty_cache()
    return None

# ========================= VALIDATION =========================
@torch.no_grad()
def run_validation(model, dataset, batch_size=VAL_BATCH_SIZE, max_samples=MAX_VAL_SAMPLES):
    model.eval()
    ds = dataset.select(range(min(max_samples, len(dataset))))
    tokenizer.padding_side = "left"
    correct, total = 0, 0
    predictions = []

    for i in range(0, len(ds), batch_size):
        batch = ds[i:i+batch_size]
        prompts_text = [build_prompt(q) for q in batch["question"]]
        gts = [a.split("####")[-1].replace(",", "").strip() for a in batch["answer"]]

        inputs = tokenizer(prompts_text, return_tensors="pt", padding=True,
                           truncation=True, max_length=512, add_special_tokens=False)
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        padded_len = inputs["input_ids"].shape[1]

        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)

        for j, out in enumerate(outputs):
            resp = tokenizer.decode(out[padded_len:], skip_special_tokens=True)
            reward = compute_reward(resp, gts[j])
            correct += reward
            total += 1
            predictions.append({"gt": gts[j], "pred": extract_answer(resp),
                                "correct": bool(reward)})

    tokenizer.padding_side = "right"
    acc = correct / total if total > 0 else 0.0
    return acc, predictions

# ========================= W&B INIT =========================
if USE_WANDB:
    import wandb
    run = wandb.init(
        project=WANDB_PROJECT,
        id=wandb_run_id, resume="allow" if wandb_run_id else None,
        config={
            "student": STUDENT_ID, "teacher": TEACHER_ID,
            "group_size": GROUP_SIZE, "lr": LR, "clip_eps": CLIP_EPS,
            "method": "DistilledRL_V12_gated",
            "temperature": TEMPERATURE,
            "entropy_threshold": ENTROPY_THRESHOLD,
            "gate_temp": GATE_TEMP,
        },
    )
    if wandb_run_id is None:
        wandb_run_id = run.id
    print(f"W&B RUN ID: {wandb_run_id}")
    print(f"W&B URL: {run.url}")

# ========================= BASELINE =========================
if base_val_score is None and start_step == 0:
    base_val_score = 0.4420
    print(f"\n=== Base student Pass@1 (pre-computed): {base_val_score:.4f} ===\n")
    if USE_WANDB:
        wandb.run.summary["base_val_pass@1"] = base_val_score

# ========================= TRAINING LOOP =========================
print(f"\nStarting V12-gated from step {start_step}...")
print(f"Gate: w = 1 + sigmoid(({ENTROPY_THRESHOLD} - H) / {GATE_TEMP}) * (rho_tilde - 1)")
n_dead_steps = 0

for step in range(start_step, MAX_STEPS):
    torch.cuda.empty_cache()
    student.train()
    torch.cuda.reset_peak_memory_stats()

    optimizer.zero_grad()
    accumulated_loss = 0.0
    grad_norm = 0.0
    post_clip_frac = 0.0

    all_frozen_chunks = []
    source_reward_sum = {}
    source_reward_count = {}

    try:
        for acc_step in range(GRAD_ACCUM_STEPS):
            result = get_informative_group(max_attempts=5)
            if result is None:
                print(f"Step {step:03d} | Accum {acc_step+1}/{GRAD_ACCUM_STEPS} | no group")
                continue

            rows, prompt_lens_list, rewards, group_ids, sources, box_rate = result

            for src, rew in zip(sources, rewards.tolist()):
                source_reward_sum[src]   = source_reward_sum.get(src, 0.0) + rew
                source_reward_count[src] = source_reward_count.get(src, 0) + 1

            advantages = group_normalize(rewards, group_ids)
            if advantages.abs().max().item() == 0:
                del rows, prompt_lens_list, rewards, group_ids, advantages
                torch.cuda.empty_cache()
                continue

            n_total = len(rows)
            chunk_size = min(TRAIN_MICROBATCH_ROWS, n_total)
            prompt_lens_tensor = torch.tensor(prompt_lens_list, device="cuda:0")

            for c in range(0, n_total, chunk_size):
                sl = slice(c, min(c + chunk_size, n_total))
                chunk_rows = rows[sl]
                chunk_prompt_lens = prompt_lens_tensor[sl]

                local_max = max(r.shape[0] for r in chunk_rows)
                chunk_input_ids = torch.full((len(chunk_rows), local_max),
                                             tokenizer.pad_token_id,
                                             dtype=torch.long, device="cuda:0")
                chunk_attention_mask = torch.zeros((len(chunk_rows), local_max),
                                                   dtype=torch.long, device="cuda:0")
                for i, row in enumerate(chunk_rows):
                    length = row.shape[0]
                    chunk_input_ids[i, :length] = row
                    chunk_attention_mask[i, :length] = 1

                A_chunk = advantages[sl].unsqueeze(1)

                with torch.no_grad():
                    old_logps_chunk, comp_mask_chunk = get_token_logps(
                        student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens
                    )
                    old_logps_chunk = old_logps_chunk.cpu()
                    comp_mask_chunk = comp_mask_chunk.cpu()
                    A_chunk_cpu = A_chunk.cpu()

                    has_positive_adv = bool((A_chunk_cpu > 0).any().item())

                    if has_positive_adv:
                        # Teacher forward on cuda:1, request per-token entropy for the gate
                        teach_logps_chunk, _, teach_tok_ent = get_token_logps(
                            teacher, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens,
                            device="cuda:1",
                            return_token_entropy=True,
                        )
                        teach_logps_chunk = teach_logps_chunk.cpu()
                        teach_tok_ent     = teach_tok_ent.cpu()

                        # rho_bar / rho_tilde (same as V12)
                        log_rho = teach_logps_chunk - old_logps_chunk
                        rho = torch.exp(log_rho)
                        rho_bar = torch.clamp(rho, 1.0 / 3.0, 3.0)
                        log_rho_bar = torch.log(rho_bar + 1e-10)
                        sum_log_rho = (log_rho_bar * comp_mask_chunk).sum(dim=1, keepdim=True)
                        count = comp_mask_chunk.sum(dim=1, keepdim=True).clamp(min=1)
                        geo_mean = torch.exp(sum_log_rho / count)
                        rho_tilde = rho_bar / (geo_mean + 1e-10)

                        # ---- Confidence gate ----
                        # c ~ 1 when teacher entropy is low (teacher confident)
                        # c ~ 0 when teacher entropy is high (teacher uncertain)
                        confidence_gate = torch.sigmoid(
                            (ENTROPY_THRESHOLD - teach_tok_ent) / GATE_TEMP
                        )

                        # ---- Gated weight: same formula as V12 when c=1 ----
                        w_chunk = torch.where(
                            A_chunk_cpu > 0,
                            1.0 + confidence_gate * (rho_tilde - 1.0),
                            torch.ones_like(rho_tilde),
                        )
                    else:
                        rho_bar  = torch.ones_like(old_logps_chunk)
                        rho_tilde = torch.ones_like(old_logps_chunk)
                        w_chunk  = torch.ones_like(old_logps_chunk)
                        confidence_gate = torch.zeros_like(old_logps_chunk)

                    # ---- gate diagnostics on positive-advantage completion tokens ----
                    pos_bool = (A_chunk_cpu > 0) & (comp_mask_chunk > 0)
                    n_pos = int(pos_bool.sum().item())
                    
                    if n_pos > 0:
                        g_vals = confidence_gate[pos_bool]
                        w_vals = w_chunk[pos_bool]
                        gate_mean_chunk          = g_vals.mean().item()
                        gate_sq_mean_chunk       = (g_vals ** 2).mean().item()
                        frac_gt_09_chunk         = (g_vals > 0.9).float().mean().item()
                        frac_lt_01_chunk         = (g_vals < 0.1).float().mean().item()
                        w_mean_chunk             = w_vals.mean().item()
                        mean_abs_w_minus_1_chunk = (w_vals - 1.0).abs().mean().item()
                    else:
                        gate_mean_chunk = gate_sq_mean_chunk = 0.0
                        frac_gt_09_chunk = frac_lt_01_chunk = 0.0
                        w_mean_chunk = 1.0
                        mean_abs_w_minus_1_chunk = 0.0

                    frozen = {
                        "input_ids":      chunk_input_ids.cpu(),
                        "attention_mask": chunk_attention_mask.cpu(),
                        "prompt_lens":    chunk_prompt_lens.cpu(),
                        "A":              A_chunk_cpu,
                        "old_logps":      old_logps_chunk,
                        "comp_mask":      comp_mask_chunk,
                        "w":              w_chunk,
                        "rho_bar":        rho_bar,
                        "rho_tilde":      rho_tilde,
                        "rewards":        rewards[sl].cpu(),
                        "adv_mean":       A_chunk.mean().item(),
                        "adv_std":        A_chunk.std().item(),
                        "pct_neg":        (A_chunk <= 0).float().mean().item(),
                        # gate diagnostics
                        "n_pos":                   n_pos,
                        "gate_mean":               gate_mean_chunk,
                        "gate_sq_mean":            gate_sq_mean_chunk,
                        "frac_gate_gt_09":         frac_gt_09_chunk,
                        "frac_gate_lt_01":         frac_lt_01_chunk,
                        "w_mean":                  w_mean_chunk,
                        "mean_abs_w_minus_1":      mean_abs_w_minus_1_chunk,
                    }

                    _, _, avg_entropy = get_token_logps(
                        student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens,
                        return_entropy=True,
                    )
                    resp_lengths = (chunk_input_ids != tokenizer.pad_token_id).sum(dim=1) - chunk_prompt_lens
                    avg_len = resp_lengths.float().mean().item()
                    mean_rho_bar   = (rho_bar   * comp_mask_chunk).sum() / comp_mask_chunk.sum().clamp(min=1)
                    mean_rho_tilde = (rho_tilde * comp_mask_chunk).sum() / comp_mask_chunk.sum().clamp(min=1)

                    frozen["avg_entropy"]    = avg_entropy
                    frozen["avg_len"]        = avg_len
                    frozen["mean_rho_bar"]   = mean_rho_bar.item()
                    frozen["mean_rho_tilde"] = mean_rho_tilde.item()
                    frozen["box_rate"]       = box_rate
                    all_frozen_chunks.append(frozen)

                del chunk_input_ids, chunk_attention_mask, chunk_prompt_lens, A_chunk
                del old_logps_chunk, comp_mask_chunk
                torch.cuda.empty_cache()

            del rows, prompt_lens_list, rewards, group_ids, advantages
            torch.cuda.empty_cache()

        if len(all_frozen_chunks) == 0:
            print(f"Step {step:03d} | no chunks collected")
            del all_frozen_chunks
            torch.cuda.empty_cache()
            continue

        total_valid_chunks = len(all_frozen_chunks)

        # ---- backward pass over each frozen chunk (true grad accumulation) ----
        for frozen in all_frozen_chunks:
            chunk_input_ids      = frozen["input_ids"].to("cuda:0")
            chunk_attention_mask = frozen["attention_mask"].to("cuda:0")
            chunk_prompt_lens    = frozen["prompt_lens"].to("cuda:0")
            chunk_old_logps      = frozen["old_logps"].to("cuda:0")
            chunk_comp_mask      = frozen["comp_mask"].to("cuda:0")
            chunk_w              = frozen["w"].to("cuda:0")
            chunk_A              = frozen["A"].to("cuda:0")

            new_logps_chunk, _ = get_token_logps(
                student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens
            )
            log_r = torch.clamp(new_logps_chunk - chunk_old_logps, -5.0, 5.0)
            r = torch.exp(log_r)
            r_clipped = torch.clamp(r, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)

            obj = torch.min(r * chunk_w * chunk_A, r_clipped * chunk_w * chunk_A)
            per_seq_obj = (obj * chunk_comp_mask).sum(dim=1) / chunk_comp_mask.sum(dim=1).clamp(min=1)
            loss = -per_seq_obj.mean() / total_valid_chunks

            if torch.isnan(loss) or torch.isinf(loss):
                raise FloatingPointError(f"NaN/Inf loss at step {step}")

            loss.backward()
            accumulated_loss += loss.item() * total_valid_chunks

            del chunk_input_ids, chunk_attention_mask, chunk_prompt_lens, chunk_old_logps
            del chunk_comp_mask, chunk_w, chunk_A, new_logps_chunk, log_r, r, r_clipped, obj, loss
            torch.cuda.empty_cache()

    except (FloatingPointError, RuntimeError) as e:
        print(f"Step {step:03d} | {e} -> discarding entire update")
        optimizer.zero_grad(set_to_none=True)
        del all_frozen_chunks
        torch.cuda.empty_cache()
        continue

    # ---- gradient sanity + optimizer step ----
    has_nan_grad = False
    for p in student.parameters():
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
            has_nan_grad = True
            break

    if has_nan_grad:
        print(f"Step {step:03d} | NaN gradient — skipping update")
        optimizer.zero_grad(set_to_none=True)
        grad_norm = 0.0
        post_clip_frac = 0.0
    else:
        grad_norm = float(torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0))
        if grad_norm > 1000:
            print(f"Step {step:03d} | spike grad_norm={grad_norm:.1f} — skipping")
            optimizer.zero_grad(set_to_none=True)
            post_clip_frac = 0.0
        else:
            optimizer.step()

            # post-update clip fraction
            post_clip_sum, post_count = 0.0, 0
            with torch.no_grad():
                for frozen in all_frozen_chunks:
                    ids_g = frozen["input_ids"].to("cuda:0")
                    am_g  = frozen["attention_mask"].to("cuda:0")
                    pl_g  = frozen["prompt_lens"].to("cuda:0")
                    ol_g  = frozen["old_logps"].to("cuda:0")
                    cm_g  = frozen["comp_mask"].to("cuda:0")

                    new_lp, _ = get_token_logps(student, ids_g, am_g, pl_g)
                    r_post = torch.exp(torch.clamp(new_lp - ol_g, -5.0, 5.0))
                    clip_m = ((r_post > 1 + CLIP_EPS) | (r_post < 1 - CLIP_EPS)).float()
                    post_clip_sum += (clip_m * cm_g).sum().item()
                    post_count    += cm_g.sum().item()

                    del ids_g, am_g, pl_g, ol_g, cm_g, new_lp, r_post, clip_m
                    torch.cuda.empty_cache()
            post_clip_frac = post_clip_sum / post_count if post_count > 0 else 0.0

            optimizer.zero_grad(set_to_none=True)

    # ---- aggregate + log ----
    n = len(all_frozen_chunks)
    avg_loss = accumulated_loss / total_valid_chunks if total_valid_chunks > 0 else 0.0

    avg_reward    = sum(f["rewards"].mean().item() for f in all_frozen_chunks) / n
    avg_adv_mean  = sum(f["adv_mean"]                for f in all_frozen_chunks) / n
    avg_adv_std   = sum(f["adv_std"]                 for f in all_frozen_chunks) / n
    avg_pct_neg   = sum(f["pct_neg"]                 for f in all_frozen_chunks) / n
    avg_entropy   = sum(f["avg_entropy"]             for f in all_frozen_chunks) / n
    avg_len       = sum(f["avg_len"]                 for f in all_frozen_chunks) / n
    avg_rho_bar   = sum(f["mean_rho_bar"]            for f in all_frozen_chunks) / n
    avg_rho_tilde = sum(f["mean_rho_tilde"]          for f in all_frozen_chunks) / n
    avg_box_rate  = sum(f["box_rate"]                for f in all_frozen_chunks) / n

    total_pos = sum(f["n_pos"] for f in all_frozen_chunks)
    
    if total_pos > 0:
        avg_gate_mean  = sum(f["gate_mean"]    * f["n_pos"] for f in all_frozen_chunks) / total_pos
        avg_gate_sq    = sum(f["gate_sq_mean"] * f["n_pos"] for f in all_frozen_chunks) / total_pos
        avg_gate_std   = max(0.0, avg_gate_sq - avg_gate_mean ** 2) ** 0.5
        avg_frac_gate_gt_09    = sum(f["frac_gate_gt_09"]    * f["n_pos"] for f in all_frozen_chunks) / total_pos
        avg_frac_gate_lt_01    = sum(f["frac_gate_lt_01"]    * f["n_pos"] for f in all_frozen_chunks) / total_pos
        avg_w_mean             = sum(f["w_mean"]             * f["n_pos"] for f in all_frozen_chunks) / total_pos
        avg_mean_abs_w_minus_1 = sum(f["mean_abs_w_minus_1"] * f["n_pos"] for f in all_frozen_chunks) / total_pos
    else:
        avg_gate_mean = avg_gate_std = 0.0
        avg_frac_gate_gt_09 = avg_frac_gate_lt_01 = 0.0
        avg_w_mean = 1.0
        avg_mean_abs_w_minus_1 = 0.0

    log_entry = {
        "step": step,
        "loss": avg_loss,
        "avg_reward": avg_reward,
        "adv_mean": avg_adv_mean,
        "adv_std": avg_adv_std,
        "rho_bar": avg_rho_bar,
        "rho_tilde": avg_rho_tilde,
        "pct_neg": avg_pct_neg,
        "resp_len": avg_len,
        "entropy": avg_entropy,
        "clip_frac": post_clip_frac,
        "grad_norm": grad_norm,
        "box_rate": avg_box_rate,
        "num_chunks": n,
        # ---- gate metrics ----
        "gate_mean":          avg_gate_mean,
        "gate_std":           avg_gate_std,
        "frac_gate_gt_09":    avg_frac_gate_gt_09,
        "frac_gate_lt_01":    avg_frac_gate_lt_01,
        "w_mean":             avg_w_mean,
        "mean_abs_w_minus_1": avg_mean_abs_w_minus_1,
    }
    for src in source_reward_count:
        log_entry[f"reward_{src}"] = source_reward_sum[src] / source_reward_count[src]
        log_entry[f"n_{src}"]      = source_reward_count[src]

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(log_entry) + "\n")
    if USE_WANDB:
        wandb.log({k: v for k, v in log_entry.items() if k != "step"}, step=step)

    per_source_str = " | ".join(
        f"{src}: rew={source_reward_sum[src]/source_reward_count[src]:.2f}"
        for src in source_reward_count
    )
    print(f"Step {step:03d} | Loss: {avg_loss:.4f} | Rew: {avg_reward:.2f} | "
          f"Adv: {avg_adv_mean:.2f}±{avg_adv_std:.2f} | Len: {avg_len:.0f} | "
          f"box: {avg_box_rate:.3f} | Ent: {avg_entropy:.3f} | clip: {post_clip_frac:.4f} | "
          f"rho: {avg_rho_bar:.2f}->{avg_rho_tilde:.2f} | "
          f"gate: {avg_gate_mean:.3f}±{avg_gate_std:.3f} | "
          f"w: {avg_w_mean:.3f} | |w-1|: {avg_mean_abs_w_minus_1:.3f} | chunks: {n}")
    if per_source_str:
        print(f"  Per-source: {per_source_str}")

    del all_frozen_chunks
    torch.cuda.empty_cache()

    # ========================= CHECKPOINT =========================
    if (step + 1) % CKPT_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"step{step + 1}")
        os.makedirs(ckpt_path, exist_ok=True)

        student.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        torch.save(optimizer.state_dict(), os.path.join(ckpt_path, "optimizer.pt"))
        torch.save(
            {"cpu": torch.get_rng_state(),
             "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
            os.path.join(ckpt_path, "rng.pt"),
        )
        if os.path.exists(LOG_FILE):
            shutil.copy(LOG_FILE, os.path.join(ckpt_path, "training_log.jsonl"))

        state = {
            "step": step + 1,
            "best_val_score": float(best_val_score),
            "best_step": int(best_step),
            "wandb_run_id": wandb_run_id,
            "base_val_score": base_val_score,
        }
        with open(os.path.join(ckpt_path, "training_state.json"), "w") as f:
            json.dump(state, f)
        print(f"  Saved checkpoint: {ckpt_path}")

        if PUSH_TO_HUB and HF_TOKEN:
            api = HfApi()
            api.create_repo(HF_HUB_ID, exist_ok=True, token=HF_TOKEN)
            api.upload_folder(
                folder_path=ckpt_path,
                repo_id=HF_HUB_ID,
                path_in_repo=f"step{step + 1}", token=HF_TOKEN,
                commit_message=f"step {step + 1} (V12-gated)",
            )
            print(f"  Pushed to HF: {HF_HUB_ID}/step{step + 1}")

    # ========================= VALIDATION =========================
    if (step + 1) % VAL_EVERY == 0:
        val_score, val_preds = run_validation(student, val_ds)
        delta = val_score - (base_val_score or 0.0)
        print(f"  >>> Validation Pass@1: {val_score:.4f} (base: {base_val_score}, delta: {delta:+.4f})")
        if USE_WANDB:
            wandb.log({"val/pass_at_1": val_score, "val/delta_vs_base": delta}, step=step)

        val_pred_file = os.path.join(CKPT_DIR, f"val_preds_step{step+1}.jsonl")
        with open(val_pred_file, "w") as f:
            for p in val_preds:
                f.write(json.dumps(p) + "\n")

        if val_score > best_val_score:
            best_val_score = val_score
            best_step = step + 1
            student.save_pretrained(BEST_DIR)
            tokenizer.save_pretrained(BEST_DIR)
            print(f"  *** New best val score! Saved to {BEST_DIR}")
            if USE_WANDB:
                wandb.run.summary["best_val_pass@1"] = best_val_score
                wandb.run.summary["best_step"] = best_step

# ========================= FINAL =========================
print("\nTraining complete. Final validation...")
final_val, _ = run_validation(student, val_ds)
print(f"Base student Pass@1:     {base_val_score}")
print(f"Final validation Pass@1: {final_val:.4f}")
print(f"Best validation Pass@1:  {best_val_score:.4f} @ step {best_step}")
print(f"Dead steps skipped:      {n_dead_steps}")

if USE_WANDB:
    wandb.log({"val/final_pass_at_1": final_val}, step=MAX_STEPS)
    wandb.run.summary["final_val_pass@1"] = final_val
    wandb.run.summary["dead_steps"] = n_dead_steps
    wandb.finish()

final_path = os.path.join(CKPT_DIR, "final")
student.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"Final checkpoint: {final_path}")